In [38]:
"""
--------------------------------------------------------------------------------
DESCRIPTION:
    Script này tạo ra bộ dữ liệu giả lập (synthetic dataset) phục vụ nghiên cứu
    dự báo nhu cầu đăng ký môn học. Dữ liệu được mô phỏng dựa trên các phân phối
    xác suất thống kê để đảm bảo tính hiện thực và nhất quán.

METHODOLOGY (PHƯƠNG PHÁP):

    1. Mô hình hóa Điểm số (Grade Modeling):
       Điểm số không ngẫu nhiên hoàn toàn mà phụ thuộc vào năng lực sinh viên và độ khó môn học.

       Final_Grade = Clamp( (Capacity * Difficulty * 100) + Noise, 0, 100 )

       Trong đó:
       - Capacity (Năng lực): [0.2, 1.0], tuân theo phân phối chuẩn hoặc lệch.
       - Difficulty (Độ khó): Hệ số điều chỉnh [0.5, 1.5].
       - Noise (Nhiễu): Biến thiên ngẫu nhiên mô phỏng các yếu tố ngoại cảnh (+/- 15).
       - Clamp: Hàm giới hạn giá trị trong khoảng [0, 100].

    2. Phân phối Năng lực (Capacity Distribution):
       - Nhóm Tồn đọng (Delayed Cohorts K17-K20):
         * 80% mẫu tuân theo phân phối thấp (0.3-0.5).
         * 20% mẫu tuân theo phân phối khá (0.6-0.8).
       - Nhóm Chính quy (Active Cohorts K21-K24):
         * Tuân theo phân phối chuẩn (Normal Distribution) với Mean=0.72, Std=0.15.

    3. Mức độ tiếng Anh chỉ sinh viên chương trình trong nước - mỗi khóa 1.5 tháng
     (ENGLISH LEVELS - each course level last 1.5 months):
        - Direct Entry: Đủ chuẩn -> Học ngay.
        - IE2 (5.0-5.5): Cần học  khóa.
        - IE1 (4.0-4.5): Cần học 2 khóa.
        - IE0 (0-4.0): Cần học 3 khóa.
    4. Population Dynamics (Động lực Dân số):
       Sử dụng mô hình "Phễu lọc" (Funnel Model) với tỷ lệ tồn tại (Retention Rate)
       giảm dần theo thâm niên để mô phỏng sinh viên bỏ học hoặc ra trường trễ.
    5. Timeline Constraint (Ràng buộc thời gian):
       - Mốc hiện tại: Đầu HK1 2025-2026.
       - Chỉ sinh điểm cho các học kỳ ĐÃ KẾT THÚC.
       - K24: Chỉ có điểm Năm 1 (HK1, HK2 2024).
       - K23: Có điểm Năm 1, 2.
    6. Pace (Tiến độ):
        - FAST (10%): Đăng ký thêm môn của kỳ sau (Học vượt tối đa 1 năm).
        - LATE (40%): Bỏ bớt môn của kỳ hiện tại (Nợ lại) hoặc Rớt.
        - NORMAL (50%): Theo đúng khung.
    7. Prerequisite Check (Kiểm tra Tiên quyết):
       - BẮT BUỘC kiểm tra quan hệ 'Prerequisite' và 'Previous' từ file SQL.
       - Chưa qua môn trước -> Không được đăng ký môn sau.
OUTPUT:
    - fake_data.sql: File SQL chứa dữ liệu giả lập.
--------------------------------------------------------------------------------
"""


import random
import re
import collections

# ==============================================================================
# 1. CẤU HÌNH HỆ THỐNG
# ==============================================================================
CURRENT_YEAR_INT = 2025
CURRENT_SEM_INT = 2  # Đang là đầu HK2 2025
SQL_INPUT_PATH = 'database_V3.sql'
SQL_OUTPUT_PATH = 'fake_data.sql'

# Mapping ID ngành -> Mã ngành
PROGRAM_MAP = { 1: 'DS', 2: 'CE', 3: 'NE', 4: 'CS' }
PROGRAM_WEIGHTS = {1: 20, 2: 25, 3: 15, 4: 40}

# Mapping ID phức tạp về ID đơn giản (1-4)
PROGRAM_ID_MAPPER = {
    1: 1, 9: 1, 49: 1,
    2: 2, 10: 2, 50: 2,
    3: 3, 11: 3, 51: 3,
    4: 4, 12: 4, 47: 4, 48: 4,
}

# Cấu hình Tiếng Anh (Tăng tỷ lệ Direct để giảm Late do tiếng Anh)
ENGLISH_LEVELS = ['Direct', 'IE2', 'IE1', 'IE0']
ENGLISH_WEIGHTS = [40, 30, 20, 10]
ENGLISH_DELAY = {'Direct': 0, 'IE2': 0, 'IE1': 1, 'IE0': 2}
PATHWAY_MAPPING = {'Direct': 3, 'IE2': 2, 'IE1': 1, 'IE0': 4}

IE_COURSES = {
    'IE0': ['ENTP00'],
    'IE1': ['ENTP01'],
    'IE2': ['ENTP02']
}

# Cấu hình độ khó
DIFF_CONFIG = {
    'VERY_HARD': (0.70, 0.90),
    'HARD':      (0.90, 1.10),
    'MEDIUM':    (1.10, 1.25),
    'EASY':      (1.25, 1.40),
    'VERY_EASY': (1.40, 1.50)
}
# List môn khó/dễ điển hình
VERY_HARD_COURSES = ['IT069','IT013', 'IT116', 'IT149', 'EE050', 'MA023', 'MA024', 'IT017', 'IT089', 'IT114', 'IT143', 'IT132', 'IT091', 'IT117', 'IT105', 'IT115']
VERY_EASY_COURSES = ['IT058', 'IT082', 'IT083', 'IT561', 'CH012']

# Dân số
BASE_INTAKE_RANGE = (220, 250)
RETENTION_RATES = {
    17: (0.01, 0.02), 18: (0.02, 0.05), 19: (0.05, 0.09), 20: (0.09, 0.15),
    21: (0.89, 0.93), 22: (0.93, 0.97), 23: (0.97, 0.99), 24: (0.99, 1.00),
    25: (1.00, 1.00)
}

# Tiến độ
PACE_TYPES = ['FAST', 'LATE', 'NORMAL']



In [39]:

# ==============================================================================
# 2. HÀM XỬ LÝ (PARSERS - ROBUST VERSION)
# ==============================================================================

def parse_course_credits(file_path):
    credits_map = {}
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            # Tìm tất cả lệnh INSERT vào bảng course
            insert_blocks = re.findall(r"INSERT INTO `course` VALUES (.*?);", content, re.DOTALL)
            for block in insert_blocks:
                block = block.replace("\n", "")
                # Regex trích xuất credit
                rows = re.findall(r"\('([^']+)',\d+,'[^']*','[^']*',(\d+),(\d+)", block)
                for cid, theory, lab in rows:
                    credits_map[cid] = int(theory) + int(lab)
            return credits_map
    except Exception as e:
        print(f"Lỗi parse credits: {e}")
        return {}

def parse_curriculum(file_path):
    # Map ID -> Code
    ID_TO_CODE = {1: 'DS', 2: 'CE', 3: 'NE', 4: 'CS'}
    curriculum = collections.defaultdict(lambda: collections.defaultdict(lambda: collections.defaultdict(list)))

    print(f"--> Đang đọc Curriculum từ '{file_path}'...")
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            # Regex mạnh để bắt khoảng trắng, xuống dòng
            pattern = r"\(\s*(\d+)\s*,\s*(\d+)\s*,\s*'([^']+)'\s*,\s*(\d+)\s*,\s*(\d+)\s*\)"
            matches = re.findall(pattern, content)

            print(f"   -> Tìm thấy {len(matches)} dòng curriculum.")

            count = 0
            for prog_id, pathway_id, course_id, sem, year_num in matches:
                prog_id, pathway_id, sem, year_num = int(prog_id), int(pathway_id), int(sem), int(year_num)

                simple_prog_id = PROGRAM_ID_MAPPER.get(prog_id)
                if not simple_prog_id: continue
                major_str = ID_TO_CODE.get(simple_prog_id)
                if not major_str: continue

                # Lấy cả Sem 3 (Hè) vào cấu trúc
                if sem in [1, 2, 3]:
                    # Tính Index phẳng: (Năm-1)*2 + Kỳ.
                    # Nếu sem=3 (Hè), tạm gán vào index của sem 2 hoặc logic riêng.
                    # Ở đây ta quy ước: Sem 1->1, Sem 2->2, Sem 3->2 (Học cùng kỳ 2 hoặc Hè sau kỳ 2)
                    # Tuy nhiên để đơn giản, ta cứ append nó vào list phẳng.
                    effective_sem = 1 if sem == 3 else sem # Coi như hè ưu tiên học bù
                    if sem == 3: pass # Logic hè xử lý riêng ở dưới hoặc gộp chung

                    flat_index = (year_num - 1) * 2 + (1 if sem == 3 else sem)
                    curriculum[major_str][pathway_id][flat_index].append(course_id)
                    count += 1

            print(f"   -> Đã load {count} môn vào bộ nhớ.")
        return curriculum
    except Exception as e:
        print(f"Lỗi parse curriculum: {e}")
        return {}

def get_difficulty_coefficient(course_id):
    if course_id in VERY_HARD_COURSES: return random.uniform(*DIFF_CONFIG['VERY_HARD'])
    if course_id in VERY_EASY_COURSES: return random.uniform(*DIFF_CONFIG['VERY_EASY'])
    random.seed(course_id)
    prefix = course_id[:2]
    coeff = 1.0
    if prefix in ['MA', 'PH', 'CH', 'EE', 'IS']: coeff = random.uniform(*DIFF_CONFIG['MEDIUM'])
    elif prefix in ['PT', 'PE']: coeff = random.uniform(*DIFF_CONFIG['VERY_EASY'])
    elif prefix in ['EN'] or course_id.startswith('ENTP'): coeff = random.uniform(*DIFF_CONFIG['EASY'])
    elif prefix == 'IT': coeff = random.uniform(*DIFF_CONFIG['HARD'])
    else: coeff = random.uniform(0.9, 1.3)
    random.seed()
    return coeff

In [40]:
# ==============================================================================
# 3. THỰC THI (EXECUTION)
# ==============================================================================
FULL_CURRICULUM = parse_curriculum(SQL_INPUT_PATH)
COURSE_CREDITS = parse_course_credits(SQL_INPUT_PATH)

if not FULL_CURRICULUM: exit()

students = []
results = []
class_sessions = set()
enrollment_batches = set()
unique_result_check = set()

# Dân số
BATCH_COUNTS = {}
total_pop = 0
for batch, (min_r, max_r) in RETENTION_RATES.items():
    intake = random.randint(*BASE_INTAKE_RANGE)
    count = int(intake * random.uniform(min_r, max_r))
    BATCH_COUNTS[batch] = count
    total_pop += count
print("\n--- SỐ SINH VIÊN GIẢ LẬP ---")
for batch, count in BATCH_COUNTS.items():
    print(f"-> K{batch}: {count} sinh viên")
print(f"--> Bắt đầu mô phỏng cho {total_pop} sinh viên...")

for batch, count in BATCH_COUNTS.items():
    for i in range(count):
        # ---------------------------------------------------------
        # 1. KHỞI TẠO SINH VIÊN
        # ---------------------------------------------------------
        prog_id_key = random.choices(list(PROGRAM_WEIGHTS.keys()), weights=list(PROGRAM_WEIGHTS.values()), k=1)[0]
        major_code = PROGRAM_MAP[prog_id_key]
        real_prog_id = prog_id_key
        eng_level = random.choices(ENGLISH_LEVELS, weights=ENGLISH_WEIGHTS, k=1)[0]
        path_db_id = PATHWAY_MAPPING[eng_level]

        # Logic Pace & Capacity (Có liên kết với English)
        if eng_level in ['IE0', 'IE1']:
            pace = random.choices(['NORMAL', 'LATE'], weights=[90, 10], k=1)[0]
            cap_mod = 0.05
        elif eng_level == 'Direct':
            pace = random.choices(['FAST', 'LATE', 'NORMAL'], weights=[20, 10, 70], k=1)[0]
            cap_mod = 0.0
        else:
            pace = random.choices(['FAST', 'LATE', 'NORMAL'], weights=[10, 10, 80], k=1)[0]
            cap_mod = 0.0

        if batch < 22:
             # Tăng capacity cho các khóa cũ để giảm tỷ lệ rớt vĩnh viễn
             base_cap = random.choices([random.uniform(0.65, 0.75), random.uniform(0.75, 0.9)], weights=[60, 40], k=1)[0]
        else:
             base_cap = random.gauss(0.80, 0.15)

        capacity = max(0.2, min(0.99, base_cap + cap_mod))

        enroll_id = f"PRO{real_prog_id}_PATH{path_db_id}"
        enrollment_batches.add(f"('{enroll_id}', {real_prog_id}, {path_db_id})")
        student_id = f"IT{major_code}IU{batch}{str(i).zfill(3)}"
        full_name = f"Student {major_code} K{batch}_{str(i).zfill(3)}"
        students.append(f"('{student_id}', '{full_name}', '{major_code}', {batch}, '{enroll_id}')")

        # ---------------------------------------------------------
        # 2. VÒNG LẶP THỜI GIAN (TIMELINE)
        # ---------------------------------------------------------
        start_year = 2000 + batch
        passed_courses = set()
        failed_courses = set() # Chứa các môn đã học nhưng < 50 điểm
        english_queue = IE_COURSES.get(eng_level, []).copy()
        current_semester_index = 0

        for year in range(start_year, CURRENT_YEAR_INT + 1):
            for sem in [1, 2, 3]:
                if year == CURRENT_YEAR_INT and sem >= CURRENT_SEM_INT: break
                if year > CURRENT_YEAR_INT: break

                acad_year_str = f"{year}-{year+1}"

                # Cập nhật Index kỳ học (chỉ tăng vào kỳ chính)
                if sem in [1, 2]: current_semester_index += 1

                # -------------------------------------------------
                # A. LẬP DANH SÁCH ƯU TIÊN (PRIORITY LIST)
                # -------------------------------------------------
                priority_1 = [] # Ưu tiên cao nhất: TA + Trả Nợ
                priority_2 = [] # Ưu tiên nhì: Môn theo lộ trình (Accumulated Sweep)
                priority_3 = [] # Ưu tiên thấp: Học vượt

                # A1. Tiếng Anh (English)
                if english_queue:
                    take_eng = 1 if sem == 3 else 2
                    for x in english_queue[:take_eng]:
                        if x not in passed_courses: priority_1.append(x)

                is_stuck_eng = (eng_level in ['IE0', 'IE1'] and len(english_queue) > 0)

                # A2. Trả nợ môn rớt (Retake)
                if not is_stuck_eng:

                    # A2. Trả nợ môn rớt (Retake)
                    if failed_courses:
                        for c in failed_courses:
                            if random.random() < 0.9: priority_1.append(c)

                    # A3. Môn theo Lộ trình & Môn sót (Accumulated Sweep)
                    limit_idx = current_semester_index
                    for idx in range(1, limit_idx + 1):
                        courses = FULL_CURRICULUM.get(major_code, {}).get(path_db_id, {}).get(idx, [])
                        for c in courses:
                            if c not in passed_courses and c not in failed_courses and c not in priority_1:
                                priority_2.append(c)

                    # A4. Học vượt (Fast Track)
                    if sem in [1, 2] and pace == 'FAST' and capacity > 0.75:
                        next_courses = FULL_CURRICULUM.get(major_code, {}).get(path_db_id, {}).get(current_semester_index + 1, [])
                        for c in next_courses:
                             if c not in passed_courses: priority_3.append(c)


                # -------------------------------------------------
                # B. ĐĂNG KÝ (ENROLLMENT)
                # -------------------------------------------------
                MAX_CREDITS = 24
                current_credit_load = 0
                final_registration = []
                seen_candidates = set()

                # Duyệt theo thứ tự ưu tiên: 1 -> 2 -> 3
                for p_list in [priority_1, priority_2, priority_3]:
                    for c in p_list:
                        if c in seen_candidates: continue
                        seen_candidates.add(c)

                        # Không check Prerequisite nữa!

                        # Check Tín chỉ
                        cred = COURSE_CREDITS.get(c, 3)
                        if current_credit_load + cred <= MAX_CREDITS:
                            final_registration.append(c)
                            current_credit_load += cred

                            # Nếu đăng ký thành công môn TA, xóa khỏi hàng đợi
                            if c in english_queue: english_queue.remove(c)

                # -------------------------------------------------
                # C. TÍNH ĐIỂM (GRADING)
                # -------------------------------------------------
                for course_id in final_registration:
                    class_id = f"{course_id}_1_{sem}_{acad_year_str}"
                    if (student_id, class_id) in unique_result_check: continue
                    unique_result_check.add((student_id, class_id))
                    class_sessions.add(f"('{class_id}', '{course_id}', {sem}, '{acad_year_str}', 1)")

                    diff = get_difficulty_coefficient(course_id)
                    curr_cap = capacity

                    # Nếu học lại thì thường điểm sẽ cao hơn do đã quen
                    if course_id in failed_courses:
                        curr_cap = min(0.99, capacity + 0.15)

                    in_class = random.randint(60, 100)
                    base = curr_cap * 100
                    final_raw = int(base * diff + random.randint(-10, 10))

                    # Random rớt ngẫu nhiên (tai nạn/bệnh tật)
                    if random.random() < 0.03: final_raw = random.randint(0, 45)

                    final = max(0, min(100, final_raw))
                    mid = max(0, min(100, final + random.randint(-10, 10)))

                    total = (in_class * 0.3) + (mid * 0.3) + (final * 0.4)

                    if total >= 50:
                        passed_courses.add(course_id)
                        # Nếu đậu môn từng rớt -> Xóa khỏi danh sách nợ
                        if course_id in failed_courses: failed_courses.remove(course_id)
                    else:
                        # Nếu rớt -> Thêm vào danh sách nợ
                        failed_courses.add(course_id)

                    results.append(f"('{student_id}', '{class_id}', {mid}, {final}, {in_class}, 4, 0, 0, 0, 0, 0, 0, 0, {round(total, 1)})")


--> Đang đọc Curriculum từ 'database_V3.sql'...
   -> Tìm thấy 5678 dòng curriculum.
   -> Đã load 2041 môn vào bộ nhớ.

--- SỐ SINH VIÊN GIẢ LẬP ---
-> K17: 2 sinh viên
-> K18: 9 sinh viên
-> K19: 19 sinh viên
-> K20: 22 sinh viên
-> K21: 217 sinh viên
-> K22: 228 sinh viên
-> K23: 222 sinh viên
-> K24: 231 sinh viên
-> K25: 247 sinh viên
--> Bắt đầu mô phỏng cho 1197 sinh viên...


In [41]:

# ==============================================================================
# 4. XUẤT FILE
# ==============================================================================
print(f"\n--> Đang xuất file '{SQL_OUTPUT_PATH}'...")
with open(SQL_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write("-- SYNTHETIC DATASET SIMPLIFIED (NO PREREQ CHECK)\n")
    f.write("USE `digit-curriculum`;\n")
    f.write("SET FOREIGN_KEY_CHECKS = 0;\n")
    f.write("TRUNCATE TABLE `result`;\n")
    f.write("TRUNCATE TABLE `student`;\n")
    f.write("TRUNCATE TABLE `enrollment_batch`;\n")
    f.write("TRUNCATE TABLE `class_session`;\n")
    f.write("SET FOREIGN_KEY_CHECKS = 1;\n\n")

    if enrollment_batches:
        f.write("INSERT INTO `enrollment_batch` (id, program_id, pathway_id) VALUES\n")
        f.write(",\n".join(enrollment_batches) + ";\n\n")

    if class_sessions:
        f.write("INSERT INTO `class_session` (id, course_id, semester, academic_year, group_theory) VALUES\n")
        f.write(",\n".join(class_sessions) + ";\n\n")

    chunk = 100
    for i in range(0, len(students), chunk):
        f.write(f"INSERT INTO `student` (id, name, major, batch, enrollment_batch_id) VALUES {','.join(students[i:i+chunk])};\n")

    res_chunk = 500
    for i in range(0, len(results), res_chunk):
        f.write(f"INSERT INTO `result` VALUES {','.join(results[i:i+res_chunk])};\n")

print("HOÀN TẤT. Dữ liệu đã được tạo thành công.")


--> Đang xuất file 'fake_data.sql'...
HOÀN TẤT. Dữ liệu đã được tạo thành công.
